In [2]:
print(df_jugadores.columns.tolist())

NameError: name 'df_jugadores' is not defined

In [1]:
"""
SIMULADOR MUNDIAL 2026 — VERSIÓN v5: INTEGRACIÓN DE DATOS HISTÓRICOS
======================================================================
Novedades respecto a v4:
  • Carga de los 3 Excel con nombres claros y legibles
  • Promedio de altura/peso por selección (nulos excluidos automáticamente)
  • Variables del output.xlsx integradas en el simulador:
      – altura promedio por equipo
      – peso promedio por equipo
      – caps promedio por equipo
      – goles internacionales promedio
      – edad promedio por mundial
      – posición dominante del plantel
  • Estadísticas históricas FIFA (wins, goles) por equipo desde 1930
  • Monte Carlo simplificado (10 000 simulaciones por defecto)
  • Red Neuronal (64 → 32 → 1)
"""

import itertools
import time
import warnings
from collections import defaultdict
from functools import lru_cache
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ═══════════════════════════════════════════════════════════════════════════════
# 1. CARGA DE ARCHIVOS EXCEL — NOMBRES CLAROS
# ═══════════════════════════════════════════════════════════════════════════════

# Rutas de los archivos (ajustar si se mueven de carpeta)
RUTA_ALTURA_PESO   = "jugadores_altura_peso.xlsx"   # altura y peso por jugador/mundial
RUTA_JUGADORES     = "output.xlsx"                  # perfil completo de jugadores históricos
RUTA_HISTORIAL_FIFA = "fifa_all_years.xlsx"         # resultados de selecciones 1930-2022

print("=" * 80)
print("📂  CARGANDO ARCHIVOS EXCEL...")
print("=" * 80)

# ── Archivo 1: Altura y Peso de jugadores por mundial ────────────────────────
df_altura_peso = pd.read_excel(RUTA_ALTURA_PESO)
# Columnas: Mundial | Nombre | Altura_cm | Peso_kg
print(f"✅  jugadores_altura_peso.xlsx  →  {len(df_altura_peso):,} filas")

# ── Archivo 2: Perfil completo de jugadores históricos ────────────────────────
df_jugadores = pd.read_excel(RUTA_JUGADORES)
# Columnas relevantes que usaremos:
#   full_name, Team, Mundial, position, Altura_cm, Edad,
#   national_team_caps, national_team_goals,
#   world_cup_participations_0_matches, world_cup_participations_0_goals
print(f"✅  output.xlsx                →  {len(df_jugadores):,} filas, {len(df_jugadores.columns)} columnas")

# ── Archivo 3: Historial FIFA por selección y año ─────────────────────────────
df_historial_fifa = pd.read_excel(RUTA_HISTORIAL_FIFA)
# Columnas: Position | Team | Games Played | Win | Draw | Loss |
#           Goals For | Goals Against | Goal Difference | Points | Year
print(f"✅  fifa_all_years.xlsx        →  {len(df_historial_fifa):,} filas")

# ═══════════════════════════════════════════════════════════════════════════════
# 2. PROCESAMIENTO: ALTURA Y PESO PROMEDIO POR SELECCIÓN
#    (usando jugadores_altura_peso.xlsx; nulos excluidos del promedio)
# ═══════════════════════════════════════════════════════════════════════════════

print("\n📐  Calculando promedios de altura y peso por selección (mundial más reciente)...")

# Unir tabla de altura/peso con tabla de jugadores para obtener el código de equipo
# Hacemos merge por nombre de jugador y año de mundial
df_merge_altura = df_altura_peso.merge(
    df_jugadores[["full_name", "Team", "Mundial"]].rename(columns={"full_name": "Nombre"}),
    on=["Nombre", "Mundial"],
    how="left"
)

# Promedio por equipo (Team) y mundial — excluyendo nulos automáticamente con mean()
promedio_altura_peso = (
    df_merge_altura
    .dropna(subset=["Team"])  # solo filas donde se identificó el equipo
    .groupby(["Team", "Mundial"])
    .agg(
        altura_promedio_cm      = ("Altura_cm"  , "mean"),   # NaN excluido por defecto
        peso_promedio_kg        = ("Peso_kg"    , "mean"),   # NaN excluido por defecto
        jugadores_con_altura    = ("Altura_cm"  , "count"),
        jugadores_con_peso      = ("Peso_kg"    , "count"),
    )
    .reset_index()
)

# Para el simulador: tomar el mundial más reciente de cada equipo
ultima_altura_por_equipo = (
    promedio_altura_peso
    .sort_values("Mundial", ascending=False)
    .groupby("Team")
    .first()
    .reset_index()
    .rename(columns={"Team": "codigo_equipo"})
)

print(f"   Equipos con datos de altura/peso: {len(ultima_altura_por_equipo)}")
print(f"   Altura promedio global: {promedio_altura_peso['altura_promedio_cm'].mean():.1f} cm")
print(f"   Peso promedio global:   {promedio_altura_peso['peso_promedio_kg'].mean():.1f} kg")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. PROCESAMIENTO: VARIABLES DEL PERFIL DE JUGADORES (output.xlsx)
# ═══════════════════════════════════════════════════════════════════════════════
#
# Variables extraídas directamente:
#   • edad_promedio          — promedio de edad del plantel en ese mundial
#   • caps_promedio          — experiencia internacional media (partidos selección)
#   • tasa_gol_seleccion     — goles por cap en selección (efectividad)
#   • altura_promedio_cm     — altura media del plantel
#
# Variables construidas por NLP sobre playing_style:
#   • pct_lider      — % de jugadores descritos como "líderes" → cohesión en campo
#   • pct_tecnico    — % de jugadores técnicos → calidad con balón
#   • pct_fisico     — % de jugadores físicos/aéreos → ventaja en duelos
#   • pct_veloz      — % de jugadores rápidos → contraataque y presión
#
# Variables construidas sobre perfil personal:
#   • pct_casados    — % casados. Correlación: +0.27 con éxito en WC
#                      (proxy de estabilidad emocional y madurez del plantel)
#   • pct_hijos      — % con hijos. Correlación: +0.32 con éxito en WC
#                      (asociado a mayor responsabilidad y motivación)
#   • pct_apodo      — % con apodo/sobrenombre. Correlación: +0.38 con éxito
#                      (proxy de jugadores icónicos / estrellas del plantel)
#   • pct_premio_ind — % con premios individuales (Balón de Oro, etc.). Corr: +0.35
#   • pct_club_elite — % de jugadores de clubes de primer nivel. Corr: +0.13
#   • pct_gano_wc    — % de jugadores que ya han ganado un Mundial. Corr: +0.82
#                      (la variable más predictiva: experiencia ganadora)
#
# ─── NOTA SOBRE TASA DE CASADOS ───────────────────────────────────────────────
# Aunque suena anecdótico, esta variable captura algo real: plantel de jugadores
# con vida estable → menos distracciones, mayor concentración y profesionalismo.
# Correlación empírica con los datos: +0.27 (moderada-alta). Se incluye con peso
# reducido para no sobreajustar.
# ──────────────────────────────────────────────────────────────────────────────

print("\n📊  Procesando variables de perfil de jugadores...")

# Convertir columnas numéricas
cols_numericas = [
    "Altura_cm", "national_team_caps", "national_team_goals", "Edad",
    "world_cup_participations_0_matches", "world_cup_participations_0_goals",
    "personal_life_children_count"
]
for col in cols_numericas:
    df_jugadores[col] = pd.to_numeric(df_jugadores[col], errors="coerce")

# ── Variables construidas a nivel jugador ────────────────────────────────────

# Tasa de gol en selección (efectividad goleadora)
df_jugadores["tasa_gol_seleccion"] = (
    df_jugadores["national_team_goals"] /
    df_jugadores["national_team_caps"].replace(0, np.nan)
)

📂  CARGANDO ARCHIVOS EXCEL...


FileNotFoundError: [Errno 2] No such file or directory: 'jugadores_altura_peso.xlsx'